### Transcripciones e Intenciones

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.bedrock_models_v2 import ModelConfig, LLMClient

In [3]:
from pprint import pprint

In [5]:
from __future__ import print_function
import time
import boto3
import pandas as pd
import requests
import json
import re
from datetime import datetime

class TranscribeIntentDetector:
    def __init__(self, region_name="us-east-1"):
        # Configuración de AWS
        self.region_name = region_name
        self.bucket_name = "cl2664-s3-demo-transbank-transcribe"
        
        # Inicializar clientes de AWS
        self.s3 = boto3.client('s3', 
                            region_name=region_name)
        
        self.transcribe = boto3.client('transcribe', 
                                     region_name=region_name)
        
        # Prompt actualizado para la detección de intención principal
        self.system_prompt = """
        Analiza la siguiente transcripción de una llamada de call center bancario e identifica la intención principal o necesidad más importante del cliente. Extrae solo el fragmento de texto donde el cliente expresa claramente qué ayuda necesita.
        
        Instrucciones:
        1. Identifica la intención o necesidad más importante expresada por el cliente.
        2. Ignora saludos, verificaciones de identidad y conversación introductoria.
        3. Extrae el texto exacto donde el cliente expresa su necesidad principal, tal cual aparece en la transcripción.
        4. Si hay múltiples solicitudes, prioriza la más importante para el cliente o la que requiere atención inmediata.
        5. Proporciona el texto extraído entre comillas.
        6. Si no identificas ninguna intención clara, responde explícitamente: "NO_INTENT_DETECTED"
        
        Ejemplo de respuesta:
        "Necesito verificar una transacción que no reconozco en mi estado de cuenta"
        """
    
    def process_audio_files(self):
        """Procesa todos los archivos de audio en el bucket especificado"""
        # Obtener la lista de archivos de audio en el bucket
        response = self.s3.list_objects_v2(Bucket=self.bucket_name)
        if 'Contents' not in response:
            print("No se encontraron archivos en el bucket.")
            return []
        
        # Filtrar por archivos de audio (ahora incluye formato .opus)
        audio_files = [content['Key'] for content in response['Contents'] 
                      if content['Key'].lower().endswith(('.m4a', '.opus'))]
        
        if not audio_files:
            print("No se encontraron archivos de audio en formatos soportados (.m4a, .opus)")
            return []
        
        # Para almacenar resultados
        results = []
        
        # Procesar cada archivo
        for audio_file in audio_files:
            print(f"\nProcesando archivo: {audio_file}")
            transcript_data = self.transcribe_audio(audio_file)
            
            if transcript_data and transcript_data.get('transcript_text'):
                # Detectar la intención y los tiempos
                intent_info = self.detect_intent(transcript_data)
                
                # Verificar si se detectó una intención
                intent_detected = (intent_info.get('intent_text', '') != 'NO_INTENT_DETECTED' and 
                                  intent_info.get('intent_text', '') != '')
                
                results.append({
                    'audio_file': audio_file,
                    'transcript': transcript_data.get('transcript_text', ''),
                    'intent_text': intent_info.get('intent_text', 'NO_INTENT_DETECTED'),
                    'intent_detected': intent_detected,
                    'start_time': intent_info.get('start_time') if intent_detected else None,
                    'end_time': intent_info.get('end_time') if intent_detected else None,
                    'confidence': intent_info.get('confidence', 0) if intent_detected else 0
                })
            else:
                # Manejar caso donde no hay transcripción
                results.append({
                    'audio_file': audio_file,
                    'transcript': 'NO_TRANSCRIPT',
                    'intent_text': 'NO_TRANSCRIPT',
                    'intent_detected': False,
                    'start_time': None,
                    'end_time': None,
                    'confidence': 0
                })
        
        # Generar un DataFrame con los resultados
        df = pd.DataFrame(results)
        
        # Guardar resultados
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_filename = f'transcriptions_with_intent_{timestamp}.csv'
        df.to_csv(output_filename, index=False, sep="|")
        print(f"\nResultados guardados en {output_filename}")
        
        return df
    
    def transcribe_audio(self, audio_file):
        """Transcribe un archivo de audio utilizando Amazon Transcribe"""
        current_date = datetime.now().strftime("%Y%m%d%H%M%S")
        job_name = f"cl2664-transbank-transcription-job-{audio_file.replace('/', '-').replace('.', '-')}-{current_date}"
        job_uri = f"s3://{self.bucket_name}/{audio_file}"
        
        print(f"Iniciando transcripción para {audio_file}...")
        
        # Determinar el formato de archivo
        file_format = 'ogg' if audio_file.lower().endswith('.opus') else 'mp4'
        
        try:
            # Iniciar el trabajo de transcripción
            self.transcribe.start_transcription_job(
                TranscriptionJobName=job_name,
                Media={'MediaFileUri': job_uri},
                MediaFormat=file_format,
                LanguageCode='es-US',  # Español de Latinoamérica
                Settings={
                    'ShowSpeakerLabels': True,
                    'MaxSpeakerLabels': 2,
                    'ShowAlternatives': True,
                    'MaxAlternatives': 2  # Obtener alternativas para mejorar coincidencia de texto
                }
            )
            
            # Esperar a que el trabajo termine
            while True:
                status = self.transcribe.get_transcription_job(TranscriptionJobName=job_name)
                if status['TranscriptionJob']['TranscriptionJobStatus'] in ['COMPLETED', 'FAILED']:
                    break
                print("Transcripción en proceso...")
                time.sleep(5)
            
            # Obtener resultados
            if status['TranscriptionJob']['TranscriptionJobStatus'] == 'COMPLETED':
                transcript_uri = status['TranscriptionJob']['Transcript']['TranscriptFileUri']
                response = requests.get(transcript_uri)
                transcript_json = json.loads(response.text)
                
                # Obtener el texto completo de la transcripción
                transcript_text = transcript_json["results"]["transcripts"][0]["transcript"]
                
                # Obtener los items con tiempos
                items = transcript_json["results"].get("items", [])
                
                print(f"Transcripción completada para {audio_file}.")
                return {
                    'transcript_text': transcript_text,
                    'items': items
                }
            else:
                print(f"Error en la transcripción de {audio_file}: {status['TranscriptionJob'].get('FailureReason', 'Razón desconocida')}")
                return None
                
        except Exception as e:
            print(f"Error iniciando o procesando la transcripción para {audio_file}: {str(e)}")
            return None
    
    def detect_intent(self, transcript_data):
        """Detecta la intención principal del cliente y los tiempos asociados"""
        try:
            from src.bedrock_models_v2 import ModelConfig, LLMClient
            
            # Inicializar el cliente LLM
            llm_client = LLMClient(model_config=ModelConfig.CLAUDE_3_7_SONNET.value)
            
            # Obtener el texto completo de la transcripción
            transcript_text = transcript_data['transcript_text']
            
            if not transcript_text or transcript_text.strip() == "":
                return {
                    'intent_text': 'NO_INTENT_DETECTED',
                    'start_time': None,
                    'end_time': None,
                    'confidence': 0
                }
            
            # Obtener la intención utilizando el LLM
            response = llm_client.invoke(f"""
            Transcripción:
            {transcript_text}
            """, system_prompt=self.system_prompt, reasoning_budget=1024)
            
            # Verificar si la respuesta es un objeto y extraer el texto
            if hasattr(response, 'content') and isinstance(response.content, str):
                response_text = response.content
            elif hasattr(response, 'to_string') and callable(getattr(response, 'to_string')):
                response_text = response.to_string()
            elif hasattr(response, '__str__'):
                response_text = str(response)
            elif isinstance(response, dict) and 'text' in response:
                response_text = response['text']
            else:
                # Si no podemos extraer el texto, lo tratamos como cadena vacía
                print("No se pudo extraer texto de la respuesta LLM:", type(response))
                response_text = ""
            
            # Extraer el texto de la intención (asumiendo que viene entre comillas)
            intent_text = self._extract_quoted_text(response_text)
            
            if not intent_text or intent_text.lower() == "no_intent_detected":
                return {
                    'intent_text': 'NO_INTENT_DETECTED',
                    'start_time': None,
                    'end_time': None,
                    'confidence': 0
                }
            
            # Encontrar los tiempos de inicio y fin de la intención
            start_time, end_time, confidence = self._find_intent_times(intent_text, transcript_data['items'])
            
            if start_time is None or end_time is None:
                print(f"No se pudieron determinar los tiempos para la intención: '{intent_text}'")
            
            return {
                'intent_text': intent_text,
                'start_time': start_time,
                'end_time': end_time,
                'confidence': confidence
            }
        except Exception as e:
            print(f"Error al detectar la intención: {str(e)}")
            import traceback
            traceback.print_exc()
            
            return {
                'intent_text': 'ERROR_DETECTING_INTENT',
                'start_time': None,
                'end_time': None,
                'confidence': 0
            }
    
    def _extract_quoted_text(self, text):
        """Extrae el texto entre comillas del resultado del LLM"""
        if not isinstance(text, str):
            print(f"Warning: expected string but got {type(text)}")
            return ""
            
        pattern = r'"([^"]*)"'
        match = re.search(pattern, text)
        if match:
            return match.group(1)
        
        # Si no hay comillas, verificar si hay "NO_INTENT_DETECTED"
        if "NO_INTENT_DETECTED" in text:
            return "NO_INTENT_DETECTED"
            
        return ""
    
    def _find_intent_times(self, intent_text, items):
        """
        Encuentra los tiempos de inicio y fin de la intención en los items de la transcripción.
        Utiliza un enfoque mejorado para encontrar coincidencias de frases.
        """
        if intent_text == "NO_INTENT_DETECTED":
            return None, None, 0
            
        # Normalizar el texto de la intención para buscar coincidencias
        normalized_intent = self._normalize_text(intent_text)
        intent_words = normalized_intent.split()
        
        if not intent_words:
            return None, None, 0
            
        # Preparar los items para búsqueda
        word_items = [item for item in items if item.get('type') == 'pronunciation']
        
        if not word_items:
            return None, None, 0
            
        # Crear una lista de palabras normalizadas de los items
        item_words = []
        for item in word_items:
            # Considerar alternativas para mejorar la coincidencia
            alternatives = item.get('alternatives', [{}])
            if alternatives:
                word = alternatives[0].get('content', '')
                item_words.append(self._normalize_text(word))
            else:
                item_words.append('')
        
        # Buscar la mejor coincidencia de la frase completa
        best_match_start = -1
        best_match_length = 0
        best_match_score = 0
        
        # Usar un enfoque de ventana deslizante para encontrar la mejor coincidencia
        for i in range(len(item_words) - len(intent_words) + 1):
            match_length = 0
            match_score = 0
            
            # Evaluar cada palabra de la intención
            for j, intent_word in enumerate(intent_words):
                if i + j < len(item_words):
                    # Calcular similitud entre palabras
                    similarity = self._word_similarity(intent_word, item_words[i + j])
                    if similarity > 0.7:  # Umbral de similitud
                        match_length += 1
                        match_score += similarity
            
            # Calcular puntuación normalizada (0-1)
            normalized_score = match_score / len(intent_words) if intent_words else 0
            
            # Actualizar mejor coincidencia si supera lo anterior
            if (match_length > best_match_length) or (match_length == best_match_length and normalized_score > best_match_score):
                best_match_start = i
                best_match_length = match_length
                best_match_score = normalized_score
        
        # Si no se encontró una coincidencia satisfactoria
        confidence_threshold = 0.7  # Al menos 70% de las palabras deben coincidir
        confidence = best_match_score
        
        if best_match_start == -1 or best_match_length < len(intent_words) * confidence_threshold:
            print(f"No se encontró una coincidencia satisfactoria para '{intent_text}'")
            print(f"Mejor coincidencia: {best_match_length}/{len(intent_words)} palabras, confianza: {best_match_score:.2f}")
            
            # Implementar una búsqueda más flexible si no hay coincidencia exacta
            if best_match_start >= 0 and best_match_score > 0.5:  # Umbral más bajo para casos difíciles
                # Usar la mejor coincidencia parcial
                start_time = float(word_items[best_match_start].get('start_time', 0))
                end_idx = best_match_start + best_match_length - 1
                if end_idx < len(word_items):
                    end_time = float(word_items[end_idx].get('end_time', 0))
                else:
                    end_time = float(word_items[-1].get('end_time', 0))
                
                return start_time, end_time, best_match_score
            return None, None, 0
        
        # Obtener los tiempos de inicio y fin
        start_time = float(word_items[best_match_start].get('start_time', 0))
        
        # Para el tiempo de fin, tomamos el tiempo de fin de la última palabra que coincide
        end_idx = best_match_start + best_match_length - 1
        if end_idx < len(word_items):
            end_time = float(word_items[end_idx].get('end_time', 0))
        else:
            end_time = float(word_items[-1].get('end_time', 0))
        
        return start_time, end_time, confidence
    
    def _normalize_text(self, text):
        """
        Normaliza el texto para mejorar las coincidencias:
        - Convierte a minúsculas
        - Elimina signos de puntuación
        - Elimina acentos
        - Elimina espacios extras
        """
        if not text:
            return ""
            
        # Convertir a minúsculas
        text = text.lower()
        
        # Eliminar signos de puntuación
        text = re.sub(r'[^\w\s]', '', text)
        
        # Normalizar caracteres acentuados
        replacements = {
            'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
            'ü': 'u', 'ñ': 'n'
        }
        for accented, normal in replacements.items():
            text = text.replace(accented, normal)
        
        # Eliminar espacios múltiples
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def _word_similarity(self, word1, word2):
        """
        Calcula la similitud entre dos palabras.
        Retorna un valor entre 0 y 1, donde 1 es coincidencia perfecta.
        """
        if not word1 or not word2:
            return 0
            
        # Coincidencia exacta
        if word1 == word2:
            return 1.0
            
        # Longitud de la palabra más larga
        max_len = max(len(word1), len(word2))
        if max_len == 0:
            return 0
            
        # Coincidencia parcial (prefijo)
        if word1.startswith(word2) or word2.startswith(word1):
            shorter = min(len(word1), len(word2))
            return shorter / max_len
            
        # Distancia de Levenshtein simplificada (número de operaciones para transformar una palabra en otra)
        common = 0
        for i in range(min(len(word1), len(word2))):
            if word1[i] == word2[i]:
                common += 1
        
        return common / max_len


# Ejemplo de uso
if __name__ == "__main__":
    try:
        # Para uso local, proporciona tus credenciales AWS
        detector = TranscribeIntentDetector(
            region_name="us-east-1"
        )
        
        # Para entornos AWS con rol IAM configurado
        # detector = TranscribeIntentDetector()
        
        results = detector.process_audio_files()
        print("\nResumen de resultados:")
        total_files = len(results) if isinstance(results, pd.DataFrame) else 0
        if total_files > 0:
            intent_detected = results['intent_detected'].sum()
            print(f"Total de archivos procesados: {total_files}")
            print(f"Archivos con intención detectada: {intent_detected} ({int(intent_detected/total_files*100)}%)")
            print(f"Archivos sin intención clara: {total_files - intent_detected} ({int((total_files-intent_detected)/total_files*100)}%)")
        else:
            print("No se procesaron archivos. Verifica la configuración y el contenido del bucket S3.")
            
    except Exception as e:
        print(f"Error general en la ejecución: {str(e)}")
        import traceback
        traceback.print_exc()


Procesando archivo: 2025-03-12_0_00e22dc3-a85e-46d4-9864-7ca7a29348f3_a9bfdf6b-0a91-4fc2-8211-8cdf3f53f892_0.opus
Iniciando transcripción para 2025-03-12_0_00e22dc3-a85e-46d4-9864-7ca7a29348f3_a9bfdf6b-0a91-4fc2-8211-8cdf3f53f892_0.opus...
Transcripción en proceso...
Transcripción completada para 2025-03-12_0_00e22dc3-a85e-46d4-9864-7ca7a29348f3_a9bfdf6b-0a91-4fc2-8211-8cdf3f53f892_0.opus.

Procesando archivo: 2025-03-12_0_01678984-2275-4b5a-9cee-3eb6d90828d8_0cbe122f-b9a3-4341-acc3-7d086466f214_0.opus
Iniciando transcripción para 2025-03-12_0_01678984-2275-4b5a-9cee-3eb6d90828d8_0cbe122f-b9a3-4341-acc3-7d086466f214_0.opus...
Transcripción en proceso...
Transcripción en proceso...
Transcripción en proceso...
Transcripción en proceso...
Transcripción en proceso...
Transcripción en proceso...
Transcripción en proceso...
Transcripción en proceso...
Transcripción en proceso...
Transcripción en proceso...
Transcripción en proceso...
Transcripción completada para 2025-03-12_0_01678984-2275-